# Análise exploratória do genoma de *Colletotrichum abscissum* Ca142

**Montagem:** GCA_023376855.1 (GenBank) · 

**Linhagem:** Ca142 · 

**Origem:** Santa Cruz do Rio Pardo, SP, flores de *Citrus sinensis*

**Referência:** Goulin et al. (2023), *Phytopathology* 113:104–107 — doi:10.1094/PHYTO-05-22-0176-A

Este caderno baixa os arquivos da montagem escolhida em `01_selecao_montagem.ipynb`, confere a integridade e faz as
primeiras contagens. A análise continua em `03_estrutura_ca142.ipynb`.

## 1. Configuração

Importamos as bibliotecas e definimos os caminhos do projeto.

In [1]:
from pathlib import Path
from collections import Counter
import hashlib
import urllib.parse
import urllib.request
import zipfile

import pandas as pd
from Bio import SeqIO

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_RAW = RAIZ / "data" / "raw"
ARQ_ZIP = DIR_RAW / "ca142.zip"
DIR_CA142 = DIR_RAW / "ca142"

## 2. Download pela API do NCBI Datasets

Montamos o endereço (URL) com o número de acesso e a lista do que queremos:

| Tipo pedido | Arquivo no pacote | Conteúdo |
|---|---|---|
| `GENOME_FASTA` | `GCA_..._genomic.fna` | sequências dos 423 scaffolds |
| `GENOME_GFF` | `genomic.gff` | anotação: posição de genes, mRNAs, éxons e CDS |
| `PROT_FASTA` | `protein.faa` | proteínas preditas (aminoácidos) |
| `CDS_FASTA` | `cds_from_genomic.fna` | sequências codificadoras (nucleotídeos, sem íntrons) |
| `SEQUENCE_REPORT` | `sequence_report.jsonl` | nome, tamanho e outras informações de cada scaffold |

O `if` evita baixar de novo toda vez que o notebook for executado: se o zip já existe, ele é reaproveitado.
Para forçar um novo download, apague `data/raw/ca142.zip`.

In [2]:
ACCESSION = "GCA_023376855.1"   # Ca142 no GenBank, obtido no caderno 01_selecao_montagem.ipynb

TIPOS = ["GENOME_FASTA", "GENOME_GFF", "PROT_FASTA", "CDS_FASTA", "SEQUENCE_REPORT"]
parametros = urllib.parse.urlencode([("include_annotation_type", t) for t in TIPOS])
url = f"https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/{ACCESSION}/download?{parametros}"

if ARQ_ZIP.exists():
    print("Zip já existe, download pulado.")
else:
    print("Baixando:", url)
    urllib.request.urlretrieve(url, ARQ_ZIP)

print(f"{ARQ_ZIP.name}: {ARQ_ZIP.stat().st_size / 1e6:.1f} MB")

Baixando: https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/GCA_023376855.1/download?include_annotation_type=GENOME_FASTA&include_annotation_type=GENOME_GFF&include_annotation_type=PROT_FASTA&include_annotation_type=CDS_FASTA&include_annotation_type=SEQUENCE_REPORT
ca142.zip: 31.1 MB


## 3. Descompactar

`zipfile` abre o pacote; `infolist()` lista o que tem dentro e `extractall()` extrai tudo para `data/raw/ca142/`.

In [3]:
with zipfile.ZipFile(ARQ_ZIP) as z:
    for info in z.infolist():
        print(f"{info.file_size / 1e6:8.2f} MB  {info.filename}")
    z.extractall(DIR_CA142)

    0.00 MB  README.md
    0.00 MB  ncbi_dataset/data/assembly_data_report.jsonl
   24.98 MB  ncbi_dataset/data/GCA_023376855.1/cds_from_genomic.fna
   54.72 MB  ncbi_dataset/data/GCA_023376855.1/GCA_023376855.1_CABS01_v0.1_genomic.fna
   31.78 MB  ncbi_dataset/data/GCA_023376855.1/genomic.gff
    8.41 MB  ncbi_dataset/data/GCA_023376855.1/protein.faa
    0.11 MB  ncbi_dataset/data/GCA_023376855.1/sequence_report.jsonl
    0.00 MB  ncbi_dataset/data/dataset_catalog.json
    0.00 MB  md5sum.txt


### Conteúdo dos arquivos extraídos

**Resultado da extração:** o zip de 31,1 MB contém 9 arquivos. Os maiores são o genoma (54,7 MB: os 54,0 milhões de
bases mais os cabeçalhos e as quebras de linha), a anotação em GFF (31,8 MB), as CDS (25,0 MB) e as proteínas (8,4 MB).

O pacote baixado do NCBI tem esta estrutura:

```
ca142/
├── README.md
├── md5sum.txt
└── ncbi_dataset/
    └── data/
        ├── assembly_data_report.jsonl
        ├── dataset_catalog.json
        └── GCA_023376855.1/
            ├── GCA_023376855.1_CABS01_v0.1_genomic.fna
            ├── genomic.gff
            ├── protein.faa
            ├── cds_from_genomic.fna
            └── sequence_report.jsonl
```

| Arquivo | Formato | Conteúdo | Uso na análise |
|---|---|---|---|
| `README.md` | texto | descrição do pacote e como citar o NCBI Datasets | consulta |
| `md5sum.txt` | texto | hash MD5 de cada arquivo | conferir a integridade (seção 4) |
| `assembly_data_report.jsonl` | JSON Lines | metadados da montagem | conferir origem, estatísticas e anotação |
| `dataset_catalog.json` | JSON | índice dos arquivos do pacote e seus tipos | localizar arquivos por programa |
| `*_genomic.fna` | FASTA (nucleotídeos) | sequência de cada scaffold | estatísticas da montagem, GC |
| `genomic.gff` | GFF3 | posição e estrutura de cada gene | estrutura gênica, distribuição dos genes |
| `protein.faa` | FASTA (aminoácidos) | proteínas preditas | tamanho e composição das proteínas |
| `cds_from_genomic.fna` | FASTA (nucleotídeos) | sequência codificadora de cada gene, já sem íntrons | desenho de dsRNA, uso de códons |
| `sequence_report.jsonl` | JSON Lines | uma linha por scaffold: nome, tamanho, papel na montagem | tabela de scaffolds |

---

#### Arquivos de controle

**`README.md`**: texto padrão do NCBI explicando a organização do pacote e como citar a fonte dos dados.

**`md5sum.txt`**: uma linha por arquivo, com o hash MD5 e o caminho. Serve só para conferir se o download veio íntegro.

**`assembly_data_report.jsonl`**: o mesmo relatório que consultamos no caderno `01_selecao_montagem.ipynb`
(submissor, tecnologia de sequenciamento, BioSample, N50, número de genes…), agora guardado junto com os dados.
O formato **JSON Lines** (`.jsonl`) tem um objeto JSON completo por linha; aqui, uma única linha, porque o pacote
tem uma única montagem.

**`dataset_catalog.json`**: um índice que diz qual arquivo contém o quê (por exemplo, que `genomic.gff` é do tipo
`GFF3` e `protein.faa` é `PROTEIN_FASTA`). É útil quando um programa precisa encontrar os arquivos sozinho.

---

#### Sequências

**`GCA_023376855.1_CABS01_v0.1_genomic.fna`: o genoma**

Arquivo FASTA com os 423 scaffolds. Cada sequência começa com uma linha de cabeçalho iniciada por `>`, com o
número de acesso do scaffold e uma descrição, seguida da sequência de bases em várias linhas:

```
>SDAQ01000001.1 Colletotrichum abscissum strain Ca142 Scaffold_001, whole genome shotgun sequence
accctaaccctaaccctaaccctaaccctaa...
```

- `A`, `C`, `G`, `T`: bases definidas;
- `N`: bases indefinidas, em geral lacunas entre contigs que foram unidos num scaffold sem que a sequência
  intermediária fosse resolvida;
- letras minúsculas indicam regiões repetitivas ou de baixa complexidade mascaradas (*soft-masking*). Este genoma
  tem trechos assim, como o início do primeiro scaffold. As análises convertem tudo para maiúsculas.

O prefixo `SDAQ01` vem do projeto WGS (*Whole Genome Shotgun*) da Ca142 no GenBank.

**`cds_from_genomic.fna`: as sequências codificadoras (CDS)**

Para cada gene, o trecho que é traduzido em proteína, do códon de início ao de parada, **já com os éxons unidos e
os íntrons removidos**. Em outras palavras, a parte codificadora do mRNA maduro. O cabeçalho traz os dados do gene
entre colchetes:

```
>lcl|SDAQ01000001.1_cds_KAI3559026.1_1 [locus_tag=CABS02_00001] [protein=hypothetical protein] [protein_id=KAI3559026.1] [location=complement(join(11380..11657,11710..13054))] [gbkey=CDS]
ATG...
```

(O exemplo é o primeiro gene da anotação, `CABS02_00001`: dois éxons na fita negativa do scaffold `SDAQ01000001.1`.)

- `locus_tag`: identificador do gene na anotação (na Ca142, tem a forma `CABS02_00001`);
- `protein`: nome do produto (muitas vezes "hypothetical protein" quando não há função conhecida);
- `protein_id`: identificador da proteína, o mesmo usado em `protein.faa`;
- `location`: posição no scaffold. `join(...)` lista os éxons e `complement(...)` indica que o gene está na fita
  negativa.

**Para o RNAi, este é o arquivo principal**: o dsRNA precisa corresponder à sequência do mRNA, e o CDS fornece essa
sequência sem os íntrons.

**`protein.faa`: as proteínas**

A tradução de cada CDS em aminoácidos (uma letra por aminoácido). O cabeçalho traz o `protein_id` e o nome do
produto:

```
>KAI3526890.1 zinc knuckle, partial [Colletotrichum abscissum]
...
```

(É o primeiro registro do arquivo. O `, partial` no fim do nome indica a proteína de um gene parcial.)

Serve para analisar tamanho e composição das proteínas e para buscas por similaridade (BLAST, domínios), que
ajudam a atribuir função aos genes.

---

#### Anotação

**`genomic.gff`: onde está cada gene e como ele é formado**

Arquivo **GFF3**: texto com uma linha por elemento do genoma e 9 colunas separadas por tabulação. Linhas iniciadas
por `#` são comentários.

| Coluna | Nome | Significado |
|---|---|---|
| 1 | seqid | scaffold onde o elemento está |
| 2 | source | programa ou banco de origem da anotação |
| 3 | type | tipo do elemento: `region`, `gene`, `mRNA`, `exon`, `CDS`… |
| 4 | start | posição inicial (contada a partir de 1) |
| 5 | end | posição final (incluída) |
| 6 | score | pontuação (`.` quando não se aplica) |
| 7 | strand | fita: `+` ou `-` |
| 8 | phase | para CDS: quantas bases (0, 1 ou 2) pular até o primeiro códon completo |
| 9 | attributes | pares `chave=valor` separados por `;` (ID, Parent, locus_tag, product…) |

Os elementos formam uma **hierarquia**, ligada pelos atributos `ID` e `Parent`:

```
region  (o scaffold inteiro)
gene
 └── mRNA          Parent = ID do gene
      ├── exon     Parent = ID do mRNA
      └── CDS      Parent = ID do mRNA
```

Com o GFF é possível medir genes, éxons e íntrons (os íntrons são os intervalos entre éxons consecutivos de um
mesmo mRNA) e ver como os genes se distribuem ao longo dos scaffolds.

**`sequence_report.jsonl`: a tabela de scaffolds**

Uma linha JSON por scaffold, com o número de acesso, o nome dado pelos autores, o tamanho e o papel na montagem
(por exemplo, `unplaced-scaffold`, um scaffold que não foi atribuído a um cromossomo). É uma forma rápida de obter
a lista de scaffolds sem ler o FASTA inteiro.

---

#### Como os arquivos se ligam

O `protein_id` é o elo entre os três arquivos de anotação: o mesmo identificador aparece no atributo do CDS em
`genomic.gff`, no cabeçalho de `cds_from_genomic.fna` e no cabeçalho de `protein.faa`. Com ele, dá para ir de um
gene à sua posição no genoma, à sua sequência codificadora e à sua proteína.

As primeiras linhas reais de cada arquivo aparecem na seção 5.

## 4. Conferir a integridade (MD5)

O NCBI inclui no pacote um arquivo `md5sum.txt` com o **hash MD5** de cada arquivo — uma "impressão digital" de 32
caracteres que muda completamente se um único byte do arquivo mudar. Recalculamos o hash de cada arquivo e comparamos
com o esperado. Todos devem dar `OK`; um `FALHOU` indica arquivo corrompido (apague o zip e baixe de novo).

A função lê o arquivo em blocos de 1 MB, para não precisar carregar arquivos grandes inteiros na memória.

In [4]:
def md5_arquivo(caminho, bloco=1 << 20):
    h = hashlib.md5()
    with open(caminho, "rb") as f:
        for pedaco in iter(lambda: f.read(bloco), b""):
            h.update(pedaco)
    return h.hexdigest()

arq_md5 = DIR_CA142 / "md5sum.txt"
if not arq_md5.exists():
    print("md5sum.txt não veio no pacote; verificação pulada.")
else:
    for linha in arq_md5.read_text().splitlines():
        if not linha.strip():
            continue
        esperado, nome = linha.split(maxsplit=1)
        nome = nome.strip().lstrip("*")
        status = "OK" if md5_arquivo(DIR_CA142 / nome) == esperado else "FALHOU"
        print(f"{status:7s} {nome}")

OK      ncbi_dataset/data/assembly_data_report.jsonl
OK      ncbi_dataset/data/GCA_023376855.1/cds_from_genomic.fna
OK      ncbi_dataset/data/GCA_023376855.1/GCA_023376855.1_CABS01_v0.1_genomic.fna
OK      ncbi_dataset/data/GCA_023376855.1/genomic.gff
OK      ncbi_dataset/data/GCA_023376855.1/protein.faa
OK      ncbi_dataset/data/GCA_023376855.1/sequence_report.jsonl
OK      ncbi_dataset/data/dataset_catalog.json


**Resultado:** os 7 arquivos listados em `md5sum.txt` deram `OK`: o download está íntegro. O `README.md` e o próprio
`md5sum.txt` não entram na lista do NCBI.

## 5. Estrutura dos arquivos

Os arquivos de dados ficam em `ncbi_dataset/data/GCA_023376855.1/`. Guardamos o caminho de cada um em uma variável
para usar no resto do notebook.

Atenção ao padrão do arquivo do genoma: usamos `GCA_023376855.1_*_genomic.fna` e não só `*_genomic.fna`,
porque este último também casaria com `cds_from_genomic.fna`.

In [5]:
for p in sorted(DIR_CA142.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:8.2f} MB  {p.relative_to(DIR_CA142)}")

DIR_DADOS = DIR_CA142 / "ncbi_dataset" / "data" / ACCESSION
ARQ_GENOMA = next(DIR_DADOS.glob(f"{ACCESSION}_*_genomic.fna"))
ARQ_GFF = DIR_DADOS / "genomic.gff"
ARQ_PROT = DIR_DADOS / "protein.faa"
ARQ_CDS = DIR_DADOS / "cds_from_genomic.fna"

print("\nGenoma:", ARQ_GENOMA.name)

    0.00 MB  README.md
    0.00 MB  md5sum.txt
   54.72 MB  ncbi_dataset/data/GCA_023376855.1/GCA_023376855.1_CABS01_v0.1_genomic.fna
   24.98 MB  ncbi_dataset/data/GCA_023376855.1/cds_from_genomic.fna
   31.78 MB  ncbi_dataset/data/GCA_023376855.1/genomic.gff
    8.41 MB  ncbi_dataset/data/GCA_023376855.1/protein.faa
    0.11 MB  ncbi_dataset/data/GCA_023376855.1/sequence_report.jsonl
    0.00 MB  ncbi_dataset/data/assembly_data_report.jsonl
    0.00 MB  ncbi_dataset/data/dataset_catalog.json

Genoma: GCA_023376855.1_CABS01_v0.1_genomic.fna


**Resultado:** os arquivos estão no lugar esperado, e o genoma foi encontrado pelo padrão de nome:
`GCA_023376855.1_CABS01_v0.1_genomic.fna` (`CABS01_v0.1` é o nome que os autores deram à montagem).

### 5.1 Primeiras linhas de cada arquivo

Antes de calcular qualquer coisa, vale olhar o começo de cada arquivo: o primeiro cabeçalho de cada FASTA e as
primeiras linhas do GFF (o primeiro gene, com seu mRNA, seus éxons e seus trechos de CDS). Assim dá para conferir os
formatos descritos em "Conteúdo dos arquivos extraídos" com os dados reais.

In [6]:
def espiar(caminho, n=1, largura=170, pular_comentarios=False):
    """Mostra as n primeiras linhas de um arquivo de texto, cortadas em `largura` caracteres."""
    print(f"--- {caminho.name}")
    with open(caminho) as f:
        mostradas = 0
        for linha in f:
            if pular_comentarios and linha.startswith("#"):
                continue
            print(linha.rstrip()[:largura])
            mostradas += 1
            if mostradas == n:
                break
    print()

espiar(ARQ_GENOMA, n=2, largura=100)
espiar(ARQ_CDS)
espiar(ARQ_PROT)
espiar(ARQ_GFF, n=7, pular_comentarios=True)

--- GCA_023376855.1_CABS01_v0.1_genomic.fna
>SDAQ01000001.1 Colletotrichum abscissum strain Ca142 Scaffold_001, whole genome shotgun sequence
accctaaccctaaccctaaccctaaccctaaccctaaccctaaccctaaccctaaccctaaccctaaccctaaccctaac

--- cds_from_genomic.fna
>lcl|SDAQ01000001.1_cds_KAI3559026.1_1 [locus_tag=CABS02_00001] [protein=hypothetical protein] [protein_id=KAI3559026.1] [location=complement(join(11380..11657,11710..130

--- protein.faa
>KAI3526890.1 zinc knuckle, partial [Colletotrichum abscissum]

--- genomic.gff
SDAQ01000001.1	Genbank	region	1	2556205	.	+	.	ID=SDAQ01000001.1:1..2556205;Dbxref=taxon:1671311;collection-date=2008;country=Brazil: Sao Paulo%2C Santa Cruz do Rio Pardo;
SDAQ01000001.1	Genbank	gene	11380	13054	.	-	.	ID=gene-CABS02_00001;Name=CABS02_00001;gbkey=Gene;gene_biotype=protein_coding;locus_tag=CABS02_00001
SDAQ01000001.1	Genbank	mRNA	11380	13054	.	-	.	ID=rna-gnl|WGS:SDAQ|mrna.CABS02_00001;Parent=gene-CABS02_00001;gbkey=mRNA;locus_tag=CABS02_00001;orig_protein_id=gnl|WG

**Resultado:** os formatos reais conferem com a descrição, e as primeiras linhas trazem quatro informações novas.

- **O genoma está mascarado.** A sequência do primeiro scaffold começa em letras minúsculas: o NCBI distribui o
  genoma com as regiões repetitivas e de baixa complexidade em minúsculas (*soft-masking*). Por isso os cadernos
  convertem as sequências para maiúsculas antes de contar bases.
- **O primeiro scaffold começa em um telômero.** As primeiras bases são `ccctaa` repetido muitas vezes. É a
  repetição telomérica dos fungos filamentosos (TTAGGG, lida aqui na fita complementar). O `Scaffold_001` chega,
  portanto, até a ponta de um cromossomo. O caderno 04 mostra que a região vizinha é rica em AT.
- **O arquivo de proteínas não está na ordem do genoma.** O primeiro registro de `protein.faa` é `KAI3526890.1`, e
  não a proteína do primeiro gene (`KAI3559026.1`). Para ligar os arquivos é preciso usar o `protein_id`, nunca a
  posição no arquivo.
- **O GFF traz dados da amostra.** A linha `region` do primeiro scaffold inclui `collection-date=2008` e o local de
  coleta: o isolado é de 2008.

As linhas seguintes do GFF mostram a hierarquia do primeiro gene: um `gene`, um `mRNA`, dois `exon` e dois `CDS`.
Na 8ª coluna das linhas de CDS aparecem as fases `0` e `2`: o segundo trecho começa no meio de um códon iniciado no
primeiro.

## 6. Estatísticas da montagem

Percorremos o FASTA do genoma com `SeqIO.parse`, que entrega um scaffold por vez, e guardamos para cada um:
tamanho e contagem de bases. Com isso calculamos:

- **número de scaffolds**, **tamanho total**, **menor** e **maior** scaffold;
- **N50**: ordenando os scaffolds do maior para o menor e somando, é o tamanho do scaffold em que a soma passa da metade
  do genoma;
- **L50**: quantos scaffolds são necessários para chegar a essa metade;
- **GC (%)**: proporção de G + C entre as bases definidas (A, C, G, T);
- **N**: bases indefinidas (lacunas entre contigs dentro de um scaffold).

Os valores devem bater com o NCBI e o artigo: 423 scaffolds, ~54,0 Mb, N50 = 321.765, L50 = 46, GC ≈ 51,1%.

In [7]:
linhas = []
for reg in SeqIO.parse(ARQ_GENOMA, "fasta"):
    s = str(reg.seq).upper()
    linhas.append({
        "scaffold": reg.id,
        "tamanho": len(s),
        "A": s.count("A"), "C": s.count("C"), "G": s.count("G"), "T": s.count("T"),
        "N": s.count("N"),
    })
df_scaf = pd.DataFrame(linhas).sort_values("tamanho", ascending=False, ignore_index=True)
# Atenção: df_scaf.T seria a transposta da tabela, por isso usamos colchetes para as colunas
acgt_scaf = df_scaf[["A", "C", "G", "T"]].sum(axis=1)
df_scaf["gc"] = (df_scaf["G"] + df_scaf["C"]) / acgt_scaf

def n50_l50(tamanhos):
    tamanhos = sorted(tamanhos, reverse=True)
    metade = sum(tamanhos) / 2
    acumulado = 0
    for i, t in enumerate(tamanhos, start=1):
        acumulado += t
        if acumulado >= metade:
            return t, i

n50, l50 = n50_l50(df_scaf.tamanho)
acgt = df_scaf[["A", "C", "G", "T"]].sum().sum()

resumo = pd.Series(dtype="object", data={
    "Scaffolds": len(df_scaf),
    "Tamanho total (pb)": df_scaf.tamanho.sum(),
    "Maior scaffold (pb)": df_scaf.tamanho.max(),
    "Menor scaffold (pb)": df_scaf.tamanho.min(),
    "N50 (pb)": n50,
    "L50": l50,
    "GC (%)": round(100 * (df_scaf["G"].sum() + df_scaf["C"].sum()) / acgt, 2),
    "Bases N": df_scaf["N"].sum(),
})
resumo

Scaffolds                   423
Tamanho total (pb)     54001896
Maior scaffold (pb)     2556205
Menor scaffold (pb)        1007
N50 (pb)                 321765
L50                          46
GC (%)                    51.12
Bases N                     846
dtype: object

In [8]:
df_scaf.head(10)   # os 10 maiores scaffolds

,scaffold,tamanho,A,C,G,T,N,gc
0,SDAQ01000001.1,2556205,617242,658894,664156,615884,29,0.517590
1,SDAQ01000002.1,1016607,256563,251767,250118,258159,0,0.493686
2,SDAQ01000003.1,906193,228396,224899,222542,230356,0,0.493759
3,SDAQ01000004.1,874598,208180,227678,228918,209812,10,0.522070
4,SDAQ01000005.1,829667,203852,209455,210231,206129,0,0.505849
5,SDAQ01000006.1,763848,186519,194921,194805,187593,10,0.510221
6,SDAQ01000007.1,741067,187289,183981,183479,186288,30,0.495873
7,SDAQ01000008.1,727520,174889,189526,189463,173642,0,0.520933
8,SDAQ01000009.1,708541,177724,177506,176464,176847,0,0.499576
9,SDAQ01000010.1,707300,175587,178653,178215,174835,10,0.504557


**Resultado:** os números calculados a partir do FASTA são **idênticos** aos do NCBI e do artigo: 423 scaffolds,
54.001.896 pb, N50 de 321.765 pb, L50 de 46, maior scaffold de 2.556.205 pb e GC de 51,1%.

- **Quase não há lacunas:** são só 846 bases `N` em 54 Mb (0,002%), em trechos curtos. Os scaffolds são quase
  inteiramente sequência contínua.
- **O maior scaffold** (2,56 Mb) tem 4,7% do genoma e é mais que o dobro do segundo (1,02 Mb).
- **O GC varia entre scaffolds:** entre os 10 maiores, vai de 49,4% a 52,2%. O caderno 03 mostra que essa variação
  é bem maior nos scaffolds menores, e o caderno 04 examina o GC ao longo de cada scaffold.
- **O menor scaffold** tem 1.007 pb, o que sugere que a montagem foi filtrada com um tamanho mínimo de 1 kb.

## 7. Proteínas e CDS

Em FASTA, cada sequência começa com uma linha `>`. Contar os registros dá o número de proteínas e de CDS.
O esperado é algo perto de **15.499** (número de genes do artigo); pode ser um pouco mais se algum gene tiver mais
de uma isoforma.

In [9]:
n_prot = sum(1 for _ in SeqIO.parse(ARQ_PROT, "fasta"))
n_cds = sum(1 for _ in SeqIO.parse(ARQ_CDS, "fasta"))
print(f"Proteínas: {n_prot:,}".replace(",", "."))
print(f"CDS:       {n_cds:,}".replace(",", "."))

Proteínas: 15.499
CDS:       15.499


**Resultado:** 15.499 proteínas e 15.499 CDS, exatamente o número de genes da anotação. Cada gene tem uma única
proteína: a anotação não inclui isoformas.

## 8. Tipos de elementos na anotação (GFF)

O GFF é um arquivo de texto com 9 colunas separadas por tabulação; cada linha descreve um elemento do genoma.
A **3ª coluna** diz o tipo (`gene`, `mRNA`, `exon`, `CDS`…). Linhas que começam com `#` são comentários.

Contar os tipos mostra como a anotação está organizada. Numa anotação típica, há um `mRNA` por `gene` (ou mais, se
houver isoformas) e vários `exon` e `CDS` por `mRNA`.

In [10]:
tipos = Counter()
with open(ARQ_GFF) as f:
    for linha in f:
        if linha.startswith("##FASTA"):   # alguns GFF trazem sequências no final
            break
        if linha.startswith("#") or not linha.strip():
            continue
        tipos[linha.split("\t")[2]] += 1

pd.Series(tipos, name="quantidade").sort_values(ascending=False)

CDS       42870
exon      42870
mRNA      15499
gene      15499
region      423
Name: quantidade, dtype: int64

**Resultado**

- **`region` = 423:** uma linha por scaffold, cobrindo o scaffold inteiro. Não é um gene; só delimita a sequência.
- **`gene` = `mRNA` = 15.499:** cada gene tem um único transcrito anotado. Também não há tRNA, rRNA, ncRNA nem
  pseudogenes: a anotação traz **só genes codificadores de proteína**.
- **`exon` = `CDS` = 42.870:** no GFF, a CDS de um gene aparece em uma linha por trecho codificador (um por éxon);
  essas 42.870 linhas pertencem às 15.499 proteínas. Como há o mesmo número de éxons e de trechos de CDS, a anotação
  provavelmente **não inclui as UTRs** (as pontas não traduzidas do mRNA): cada éxon corresponderia exatamente a um
  trecho codificador. O caderno 03 confirma isso comparando as coordenadas.
- **Éxons por gene:** 42.870 ÷ 15.499 = 2,77 em média.
- **Íntrons:** um mRNA com *n* éxons tem *n* − 1 íntrons, então são 42.870 − 15.499 = **27.371 íntrons** (1,77 por
  gene).

**Conclusão do caderno:** os arquivos estão íntegros e são coerentes entre si e com o que o NCBI e o artigo informam.
O próximo caderno, `03_estrutura_ca142.ipynb`, passa das contagens às distribuições: contiguidade, estrutura dos
genes, íntrons e proteínas.